![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 39 -- Competition: Hidden Hate

A social network wants to find posts that spread hate without a single slur: the hate hides in hints, jokes, questions and hashtags. Your goal is to label every post of the test file as toxic (1) or not toxic (0), and to beat the starter solution below. Your file is scored on Kaggle with accuracy, the share of posts labelled right.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Real posts from a social network, already in lower case, with punctuation split off by spaces and with links and user names removed. Some posts are offensive: they were collected because they carry hate. The setup cell leaves four files in `DATA_DIR`: `train.csv` (`id`, `text`, `label`: 1 toxic, 0 not toxic), `test.csv` (`id`, `text`: the posts you label), `unlabeled.csv` (`text`: more posts from the same network, without a label) and `sample_submission.csv` (the format of the file you hand in).

Your work is checked as you go, and the clock started at the opening time, so start at once and run every checkpoint cell.

---
# Setup

Run the four cells below in order. The first one installs the checkpoint library where it is missing. The second one opens your session with the scorer: replace `PASTE-YOUR-ID` by your own student id before you run it. From that cell on, this notebook records cell runs, cell source and errors for assessment. The third one imports the libraries and the fourth one gets the data.

Your id is yours alone. Do not share a copy of this notebook that holds it.

In [ ]:
# Installs the checkpoint library where it is missing. On Colab that takes a few seconds.
import importlib.util

if importlib.util.find_spec("olympiad") is None:
    !pip -q install olympiad-lab
    importlib.invalidate_caches()


In [ ]:
import os
from olympiad import ck

STUDENT_ID = os.environ.get("OLYMPIAD_STUDENT_ID", "PASTE-YOUR-ID")
ck.start(lab="nlp-comp", student_id=STUDENT_ID)


In [ ]:
import copy
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

torch.manual_seed(0)
print("torch", torch.__version__)


In [ ]:
import os

DATA_DIR = os.environ.get("OLYMPIAD_DATA_DIR")      # set when the instructor runs the notebook again
if not DATA_DIR:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download("sattamjaltwaim/ai-olympiad-l1-2027-week-6-competition-data")
print("Data folder:", DATA_DIR)
print(sorted(os.listdir(DATA_DIR))[:20])


---
# How this notebook works

Every olympiad task comes with a **starter solution**: a notebook that reads the files and hands in a valid file. Its score is the **baseline**, and your points count from it upward. Here the starter is already written into this notebook, part by part, with a checkpoint after each part, and it runs from top to bottom as it is. Run it first, read every cell, answer the task reading, and put the starter's file on Kaggle. Then the competition is yours: beat the baseline.

---
# Part 1 -- Load and look

Every task begins the same way: get the files into memory, then look at the posts before you build anything.

Read the four files.

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
unlabeled = pd.read_csv(os.path.join(DATA_DIR, "unlabeled.csv"))
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
print(train.shape, test.shape, unlabeled.shape, sample.shape)


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.load(train=train, test=test, unlabeled=unlabeled, sample=sample)


Look at the posts: a few rows, the posts per class, the words per post, and five toxic posts.

In [ ]:
first_rows = train.head(5)
n_rows = len(train)
print(first_rows)
print(train["label"].value_counts())                         # posts per class
print(train["text"].str.split().str.len().describe())         # words per post
for text in train.loc[train["label"] == 1, "text"].head(5):   # five toxic posts
    print("-", text)


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.look(first_rows, size=n_rows)


---
# Part 2 -- Read the task

Before any model comes the task reading: what the metric is, what two answers without a model would score, what the metric rewards, which format the file must have and what you noticed about the task. It is sent by the cell after Task 5. It earns full credit if it arrives by minute 25; after that it earns less, down to nothing at the end of the lab. It is scored after the lab, so no tick appears. Each answer cell holds a placeholder (`None`) so that the notebook runs from top to bottom: replace it with your answer and run the cells again.

The scorer of this task, in one line of scikit-learn. Run the cell; you will use `score(y_true, y_pred)` all afternoon.

In [ ]:
def score(y_true, y_pred):
    """The score of a file on this task, as Kaggle computes it."""
    return accuracy_score(y_true, y_pred)


print(score([0, 1, 1, 0, 0], [0, 1, 0, 0, 1]))


## Task 1: What the scorer computes

The cell above is the scorer of this task. Which sentence says what it computes?

- **a** The share of posts whose label is right: every post counts the same.
- **b** The F1 score of the toxic class: how well the toxic posts are found, from precision and recall.
- **c** The mean of the recall of the two classes: each class counts the same, whatever its size.

Store the letter in `formula`.

In [ ]:
formula = None          # replace None by the letter of your answer, in quotes: "a", "b" or "c"


## Task 2: Two answers without a model

What would two answers that need no model score? `all_0` labels every post not toxic, and `all_1` labels every post toxic. Score both on the labels of `train` with `score` and store the two numbers in the dictionary `trivial` under these two names.

In [ ]:
trivial = {"all_0": None, "all_1": None}      # replace each None by a score
print(trivial)


## Task 3: What the metric rewards

The test file has 1,000 posts, 300 of them toxic. File A labels 150 posts toxic, all of them right, and every other post not toxic. File B labels 400 posts toxic: it finds 250 of the toxic posts but also labels 150 harmless posts toxic. Which file does the metric prefer?

- **a** File B: it finds 250 of the toxic posts, and file A finds only 150.
- **b** File A: it makes 150 mistakes, file B makes 200, and every mistake costs the same.
- **c** They tie: each file finds at least half of the toxic posts.

Store the letter in `rewards`.

In [ ]:
rewards = None          # replace None by the letter of your answer, in quotes


## Task 4: The format of the file

Store the format rules in the dictionary `format_rules`: `n_rows`, the number of rows your file must have, and `id_col`, the name of its id column. Read both from the tables you loaded; do not type the number.

In [ ]:
format_rules = {"n_rows": None, "id_col": None}      # replace each None
print(format_rules)


## Task 5: What you noticed

Write in `facts`, in one to three sentences of your own, what you noticed about this task that its statement does not say and that could change your score.

In [ ]:
facts = ""              # write what you noticed between the quotes


Run the cell to send your task reading. It answers with one line and shows no score.

In [ ]:
ck.reading(formula=formula, trivial=trivial, rewards=rewards, format=format_rules, facts=facts)


---
# Part 3 -- The starter solution

The baseline, written out. It turns every post into word numbers, reads them with a plain recurrent network (Day 37), and writes a file. Run each cell and its checkpoint; change nothing yet.

**SPLIT.** A validation set, held out from the labelled posts.

In [ ]:
# Hold out a fifth of the labelled posts: the model never trains on them, so they tell how it does on new posts.
tr, va = train_test_split(train, test_size=0.2, random_state=0, stratify=train["label"])
train_ids, val_ids = tr["id"], va["id"]
print(len(tr), "posts to train on,", len(va), "to validate on")


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.split(train_ids=train_ids, val_ids=val_ids)


**BATCH.** From words to numbers, and one batch.

In [ ]:
# The vocabulary: every word seen at least MIN_COUNT times in the training part gets a number.
# 0 fills the end of short posts, 1 stands for any other word.
MIN_COUNT, MAX_WORDS = 5, 80
counts = Counter(word for text in tr["text"] for word in text.split())
vocab = {"<pad>": 0, "<unk>": 1}
for word, count in counts.items():
    if count >= MIN_COUNT:
        vocab[word] = len(vocab)


def encode(texts):
    """One row of MAX_WORDS word numbers per post: its words in order, then zeros."""
    ids = torch.zeros(len(texts), MAX_WORDS, dtype=torch.long)
    for row, text in enumerate(texts):
        numbers = [vocab.get(word, 1) for word in text.split()][:MAX_WORDS]
        ids[row, :len(numbers)] = torch.tensor(numbers, dtype=torch.long)
    return ids


X_tr, X_va, X_test = encode(tr["text"]), encode(va["text"]), encode(test["text"])
y_tr, y_va = torch.tensor(tr["label"].values), torch.tensor(va["label"].values)
xb, yb = X_tr[:512], y_tr[:512]                 # one batch, as the loop below will see it
print(len(vocab), "words in the vocabulary;", "one batch:", tuple(xb.shape), tuple(yb.shape))


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.batch(xb, yb)


**MODEL.** The network, as a class with `__init__` and `forward`.

In [ ]:
class StarterRNN(nn.Module):
    """Word numbers in, two scores out (not toxic, toxic): a plain RNN that reads a post word by word."""

    def __init__(self, vocab_size, embed_dim=300, hidden_dim=300, layers=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, num_layers=layers, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, ids):
        out, _ = self.rnn(self.dropout(self.embedding(ids)))   # (posts, words, hidden_dim)
        lengths = (ids != 0).sum(dim=1).clamp(min=1)            # the real words of every post
        last = out[torch.arange(len(ids)), lengths - 1]         # the state after the last real word
        return self.fc(self.dropout(last))


torch.manual_seed(0)
model = StarterRNN(len(vocab))
print(model(xb).shape)


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.model(model, xb)


**LOOP.** The training loop, written out. It takes a few seconds.

In [ ]:
# The four moves, epoch after epoch; the weights with the lowest validation loss are kept.
EPOCHS, BATCH_SIZE = 10, 512
torch.manual_seed(0)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
loss_fn = nn.CrossEntropyLoss()
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
losses, best_val_loss = [], float("inf")
for epoch in range(EPOCHS):
    model.train()
    for x, y in loader:
        loss = loss_fn(model(x), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(X_va), y_va).item()
    if val_loss < best_val_loss:
        best_val_loss, best_state = val_loss, copy.deepcopy(model.state_dict())
    print(f"epoch {epoch}: training loss {np.mean(losses[-len(loader):]):.3f}, validation loss {val_loss:.3f}")
model.load_state_dict(best_state)


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.loop(losses)


**EVAL.** The task's metric on the validation set.

In [ ]:
model.eval()
with torch.no_grad():
    val_pred = model(X_va).argmax(dim=1).numpy()
val_score = score(va["label"], val_pred)
starter_val_score = val_score                   # kept, to compare your own models with
print("validation accuracy of the starter:", round(val_score, 4))


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score, metric="accuracy")


**SUBMIT.** The file, in the format of `sample`.

In [ ]:
with torch.no_grad():
    test_pred = model(X_test).argmax(dim=1).numpy()
submission = pd.DataFrame({"id": test["id"], "label": test_pred})
submission.to_csv("submission.csv", index=False)
print(submission.shape, "share labelled toxic:", round(submission["label"].mean(), 3))


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


**Now put this file on the Kaggle leaderboard.** The cell above checked `submission.csv`; the score comes from Kaggle. On Colab, download the file first: open the Files panel (the folder icon on the left), find `submission.csv`, click the three dots beside it, then Download. On the competition's Kaggle page click **Submit Prediction**, choose the file and click **Submit**. Its score is about the baseline: every point above it is yours. Do the same each time you write a better file.

---
# Part 4 -- Beat the baseline

The rest of the competition is yours. Use what you learned on Days 36, 37 and 38 and in their labs. Change one thing at a time, compare every model on the validation set `va` with `score`, and hand in a new file (with the checkpoint here, and on Kaggle) each time your validation score rises. Your last notebook must still run from top to bottom in under 20 minutes, and the file it writes last is the one you choose for your final score on Kaggle.

## Task 6: A better model

Train a model of your choice on the training part `tr` and store its accuracy on the validation part `va` in `val_score`: one number, computed with `score`. Print it next to `starter_val_score`. Add as many cells as you need above this one.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.eval(val_score, metric="accuracy")


## Task 7: Its file

Write `submission.csv` again with the labels your better model gives the posts of `test`, in the format of `sample`. Then run the checkpoint and put the file on Kaggle.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


## Task 8: Your best file

Keep improving until the end, one change at a time, and let the validation set decide what stays. End this cell by writing `submission.csv` with your best labels. Your last accepted file is your answer, so hand in a new file only when you expect it to be better.

In [ ]:
# Your code here


Run the checkpoint cell. It prints a tick or a cross and one sentence that says why.

In [ ]:
ck.submit("submission.csv")


---
## Task 9: Reflect

Which change raised your validation score the most, and by how much? Did the public leaderboard agree with your validation set? What would you try with one more hour?

*Your answers here*

---
# Finish

Run the last cell when you are done. It sends what is still waiting and hands in this notebook.

In [ ]:
ck.finish()

---
## Conclusion

- **Read the task**: the scorer read as code, two answers without a model, what accuracy rewards, and the format of the file, before any model.
- **A starter solution**: the baseline read part by part, its file on the leaderboard in the first minutes.
- **Beat the baseline**: models from Days 36 to 38 compared on one validation set, one change at a time.
- **A competition file**: the best model trained on every labelled post, its file chosen for the final score.

---

Made By **Sattam Altwaim**